# 🧠 Spaceship Titanic — Model Training Report

This notebook documents the full training pipeline:
1. Data preparation
2. Optuna hyperparameter search
3. Cross-validation loss & AUC curves
4. Confusion matrix & classification report
5. Feature importance
6. MLflow experiment summary

In [ ]:
import os, sys, warnings, logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import tensorflow as tf
import optuna
import mlflow
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve, precision_recall_curve
)

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
sys.path.insert(0, '../src')
from preprocessing import engineer_features, build_preprocessor
from model import build_mlp, build_wide_deep

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

sns.set_theme(style='darkgrid')
plt.rcParams.update({'figure.dpi': 130, 'axes.facecolor': '#0f1c35',
                     'figure.facecolor': '#0a0e1a', 'text.color': '#e0e6f0',
                     'axes.labelcolor': '#7eb8f7', 'xtick.color': '#aaa',
                     'ytick.color': '#aaa', 'axes.edgecolor': '#1e3a6e',
                     'grid.color': '#1e3a6e'})

print('TensorFlow:', tf.__version__)
print('Optuna:', optuna.__version__)
print('MLflow:', mlflow.__version__)

## 1. Load & Prepare Data

In [ ]:
train = pd.read_csv('../data/train.csv')
test  = pd.read_csv('../data/test.csv')

train = engineer_features(train)
test  = engineer_features(test)

TARGET   = 'Transported'
DROP     = [TARGET, 'PassengerId', 'Name']
X        = train.drop(columns=DROP, errors='ignore')
y        = train[TARGET].astype(int).values
X_test   = test.drop(columns=['PassengerId','Name'], errors='ignore')

print(f'Features: {X.shape[1]} | Train: {len(y)} | Positive rate: {y.mean():.3f}')
X.head(3)

## 2. Optuna Hyperparameter Search

In [ ]:
def make_objective(X, y):
    def objective(trial):
        params = {
            'n_layers':   trial.suggest_int('n_layers', 2, 5),
            'units':      trial.suggest_categorical('units', [64, 128, 256]),
            'dropout':    trial.suggest_float('dropout', 0.1, 0.5),
            'lr':         trial.suggest_float('lr', 1e-4, 1e-2, log=True),
            'batch_size': trial.suggest_categorical('batch_size', [128, 256]),
            'activation': trial.suggest_categorical('activation', ['relu', 'swish']),
            'l2_reg':     trial.suggest_float('l2_reg', 1e-5, 1e-2, log=True),
        }
        skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
        scores = []
        for tr_idx, val_idx in skf.split(X, y):
            X_tr, X_val = X[tr_idx], X[val_idx]
            y_tr, y_val = y[tr_idx], y[val_idx]
            prep = build_preprocessor()
            X_tr_t  = prep.fit_transform(X_tr)
            X_val_t = prep.transform(X_val)
            model = build_mlp(X_tr_t.shape[1], params)
            model.fit(X_tr_t, y_tr, validation_data=(X_val_t, y_val),
                      epochs=25, batch_size=params['batch_size'],
                      callbacks=[tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)],
                      verbose=0)
            preds = model.predict(X_val_t, verbose=0).ravel()
            scores.append(roc_auc_score(y_val, preds))
        return np.mean(scores)
    return objective

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(make_objective(X.values, y), n_trials=20, show_progress_bar=True)

best_params = study.best_params
print(f'\nBest AUC: {study.best_value:.4f}')
print('Best params:', best_params)

### Optuna — Optimization History

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Optimization history
trials_df = study.trials_dataframe()
axes[0].plot(trials_df.index, trials_df['value'], 'o-', color='#4a90d9', alpha=0.6, markersize=4, label='Trial AUC')
best_so_far = trials_df['value'].cummax()
axes[0].plot(trials_df.index, best_so_far, color='#00e676', linewidth=2, label='Best so far')
axes[0].set_xlabel('Trial')
axes[0].set_ylabel('AUC')
axes[0].set_title('Optuna Optimization History', color='#7eb8f7')
axes[0].legend(facecolor='#0f1c35', edgecolor='#1e3a6e', labelcolor='white')

# Param importance (approximated)
param_importance = optuna.importance.get_param_importances(study)
params_sorted = dict(sorted(param_importance.items(), key=lambda x: x[1]))
axes[1].barh(list(params_sorted.keys()), list(params_sorted.values()), color='#4a90d9')
axes[1].set_title('Hyperparameter Importance', color='#7eb8f7')
axes[1].set_xlabel('Importance Score')

plt.tight_layout()
plt.savefig('../models/optuna_study.png', dpi=130, bbox_inches='tight', facecolor='#0a0e1a')
plt.show()

## 3. Cross-Validation Training — Loss & AUC Curves

In [ ]:
N_FOLDS = 5
skf     = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

fold_histories = []
fold_aucs      = []
fold_accs      = []
oof_preds      = np.zeros(len(y))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X.values, y)):
    print(f'\n── Fold {fold+1}/{N_FOLDS} ──────────────────────')
    X_tr, X_val = X.values[tr_idx], X.values[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]

    prep    = build_preprocessor()
    X_tr_t  = prep.fit_transform(X_tr)
    X_val_t = prep.transform(X_val)

    model = build_mlp(X_tr_t.shape[1], best_params)
    history = model.fit(
        X_tr_t, y_tr,
        validation_data=(X_val_t, y_val),
        epochs=80,
        batch_size=best_params['batch_size'],
        callbacks=[
            tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
            tf.keras.callbacks.ReduceLROnPlateau(patience=5, factor=0.5, verbose=0),
        ],
        verbose=1,
    )
    fold_histories.append(history.history)

    preds = model.predict(X_val_t, verbose=0).ravel()
    oof_preds[val_idx] = preds
    auc = roc_auc_score(y_val, preds)
    acc = accuracy_score(y_val, (preds > 0.5).astype(int))
    fold_aucs.append(auc)
    fold_accs.append(acc)
    print(f'  → AUC={auc:.4f}  ACC={acc:.4f}')

print(f'\n✅ Mean AUC={np.mean(fold_aucs):.4f} ± {np.std(fold_aucs):.4f}')
print(f'   Mean ACC={np.mean(fold_accs):.4f} ± {np.std(fold_accs):.4f}')

### Loss Curves per Fold

In [ ]:
fig, axes = plt.subplots(2, N_FOLDS, figsize=(18, 7))
colors = ['#4a90d9','#e67e22','#2ecc71','#9b59b6','#e74c3c']

for fold_idx, hist in enumerate(fold_histories):
    ax_loss = axes[0, fold_idx]
    ax_auc  = axes[1, fold_idx]
    c = colors[fold_idx]

    ax_loss.plot(hist['loss'],     color=c,       label='Train', linewidth=1.5)
    ax_loss.plot(hist['val_loss'], color=c, linestyle='--', label='Val', alpha=0.8)
    ax_loss.set_title(f'Fold {fold_idx+1} — Loss', color='#7eb8f7', fontsize=10)
    ax_loss.set_ylabel('BCE Loss' if fold_idx==0 else '')
    ax_loss.legend(fontsize=7, facecolor='#0f1c35', labelcolor='white', edgecolor='none')

    auc_key = [k for k in hist.keys() if 'auc' in k.lower() and 'val' not in k.lower()]
    val_auc_key = [k for k in hist.keys() if 'auc' in k.lower() and 'val' in k.lower()]
    if auc_key:
        ax_auc.plot(hist[auc_key[0]],     color=c,       label='Train', linewidth=1.5)
        ax_auc.plot(hist[val_auc_key[0]], color=c, linestyle='--', label='Val', alpha=0.8)
    ax_auc.set_title(f'Fold {fold_idx+1} — AUC', color='#7eb8f7', fontsize=10)
    ax_auc.set_ylabel('AUC' if fold_idx==0 else '')
    ax_auc.set_xlabel('Epoch')
    ax_auc.legend(fontsize=7, facecolor='#0f1c35', labelcolor='white', edgecolor='none')

plt.suptitle('Training Curves — All Folds', color='#7eb8f7', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../models/loss_curves.png', dpi=130, bbox_inches='tight', facecolor='#0a0e1a')
plt.show()

### Fold AUC Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar([f'Fold {i+1}' for i in range(N_FOLDS)], fold_aucs,
              color=['#4a90d9','#e67e22','#2ecc71','#9b59b6','#e74c3c'],
              width=0.5, edgecolor='none')
ax.axhline(np.mean(fold_aucs), color='#00e676', linestyle='--', linewidth=2,
           label=f'Mean AUC = {np.mean(fold_aucs):.4f}')
ax.set_ylim(0.75, 0.95)
ax.set_title('AUC per Fold', color='#7eb8f7')
ax.set_ylabel('ROC-AUC')
for bar, val in zip(bars, fold_aucs):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
            f'{val:.4f}', ha='center', va='bottom', fontsize=9, color='white')
ax.legend(facecolor='#0f1c35', edgecolor='#1e3a6e', labelcolor='white')
plt.tight_layout()
plt.savefig('../models/fold_auc.png', dpi=130, bbox_inches='tight', facecolor='#0a0e1a')
plt.show()

## 4. Confusion Matrix & Classification Report

In [ ]:
oof_binary = (oof_preds > 0.5).astype(int)
cm = confusion_matrix(y, oof_binary)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion matrix heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Not Transported','Transported'],
            yticklabels=['Not Transported','Transported'],
            linewidths=0.5, linecolor='#1e3a6e')
axes[0].set_title('Confusion Matrix (OOF)', color='#7eb8f7')
axes[0].set_ylabel('True Label', color='#7eb8f7')
axes[0].set_xlabel('Predicted Label', color='#7eb8f7')

# ROC Curve
fpr, tpr, _ = roc_curve(y, oof_preds)
auc_score   = roc_auc_score(y, oof_preds)
axes[1].plot(fpr, tpr, color='#4a90d9', linewidth=2, label=f'ROC (AUC={auc_score:.4f})')
axes[1].plot([0,1],[0,1], 'k--', alpha=0.4, label='Random')
axes[1].fill_between(fpr, tpr, alpha=0.1, color='#4a90d9')
axes[1].set_title('ROC Curve (OOF)', color='#7eb8f7')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend(facecolor='#0f1c35', edgecolor='#1e3a6e', labelcolor='white')

plt.tight_layout()
plt.savefig('../models/confusion_roc.png', dpi=130, bbox_inches='tight', facecolor='#0a0e1a')
plt.show()

print('\n── Classification Report ──────────────────────')
print(classification_report(y, oof_binary, target_names=['Not Transported','Transported']))

### Precision-Recall Curve

In [ ]:
precision, recall, thresholds = precision_recall_curve(y, oof_preds)
f1_scores = 2 * precision * recall / (precision + recall + 1e-9)
best_thresh = thresholds[np.argmax(f1_scores[:-1])]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(recall, precision, color='#e67e22', linewidth=2)
axes[0].fill_between(recall, precision, alpha=0.1, color='#e67e22')
axes[0].set_title('Precision-Recall Curve', color='#7eb8f7')
axes[0].set_xlabel('Recall')
axes[0].set_ylabel('Precision')

axes[1].plot(thresholds, f1_scores[:-1], color='#2ecc71', linewidth=2)
axes[1].axvline(best_thresh, color='#e74c3c', linestyle='--',
                label=f'Best threshold = {best_thresh:.2f}')
axes[1].set_title('F1 Score vs Threshold', color='#7eb8f7')
axes[1].set_xlabel('Threshold')
axes[1].set_ylabel('F1 Score')
axes[1].legend(facecolor='#0f1c35', edgecolor='#1e3a6e', labelcolor='white')

plt.tight_layout()
plt.savefig('../models/pr_curve.png', dpi=130, bbox_inches='tight', facecolor='#0a0e1a')
plt.show()
print(f'Optimal threshold: {best_thresh:.4f}  →  F1={f1_scores[:-1].max():.4f}')

## 5. Prediction Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# OOF probability distribution
axes[0].hist(oof_preds[y==0], bins=40, alpha=0.7, color='#ff5252', label='Not Transported', density=True)
axes[0].hist(oof_preds[y==1], bins=40, alpha=0.7, color='#00e676', label='Transported',     density=True)
axes[0].axvline(0.5, color='white', linestyle='--', alpha=0.6, label='Threshold=0.5')
axes[0].set_title('OOF Probability Distribution', color='#7eb8f7')
axes[0].set_xlabel('Predicted Probability')
axes[0].set_ylabel('Density')
axes[0].legend(facecolor='#0f1c35', edgecolor='#1e3a6e', labelcolor='white')

# Calibration
from sklearn.calibration import calibration_curve
fraction_pos, mean_pred = calibration_curve(y, oof_preds, n_bins=15)
axes[1].plot(mean_pred, fraction_pos, 's-', color='#4a90d9', linewidth=2, label='Model')
axes[1].plot([0,1],[0,1], 'k--', alpha=0.5, label='Perfect calibration')
axes[1].set_title('Calibration Curve', color='#7eb8f7')
axes[1].set_xlabel('Mean Predicted Probability')
axes[1].set_ylabel('Fraction of Positives')
axes[1].legend(facecolor='#0f1c35', edgecolor='#1e3a6e', labelcolor='white')

plt.tight_layout()
plt.savefig('../models/prediction_dist.png', dpi=130, bbox_inches='tight', facecolor='#0a0e1a')
plt.show()

## 6. Training Summary

In [ ]:
summary = pd.DataFrame({
    'Fold':         [f'Fold {i+1}' for i in range(N_FOLDS)] + ['MEAN ± STD'],
    'AUC':          fold_aucs + [f'{np.mean(fold_aucs):.4f} ± {np.std(fold_aucs):.4f}'],
    'Accuracy':     fold_accs + [f'{np.mean(fold_accs):.4f} ± {np.std(fold_accs):.4f}'],
})
summary.style.set_properties(**{'background-color': '#0f1c35', 'color': '#e0e6f0'})

In [ ]:
print('═'*50)
print('  TRAINING COMPLETE — FINAL RESULTS')
print('═'*50)
print(f'  OOF AUC:      {roc_auc_score(y, oof_preds):.4f}')
print(f'  OOF Accuracy: {accuracy_score(y, oof_binary):.4f}')
print(f'  Best Params:  {best_params}')
print('═'*50)
print()
print('Saved plots:')
for f in ['optuna_study.png','loss_curves.png','fold_auc.png','confusion_roc.png','pr_curve.png','prediction_dist.png']:
    print(f'  ../models/{f}')